# Film selection: five-decade check

Read one TMDB results page per decade, from 1970 to 2019. Use TMDB rating >= 7.0 and vote count >= 1,000, as chosen by the student. Sort by vote count to see familiar films first. This is a small coverage check, not the final sample.

Start Jupyter in the project root. Run top to bottom. Saved responses are used by default. Live requests read the token from the local `.env` file. Never print that file or the request headers.

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import requests
import pandas as pd
import json
import time
import hashlib

folder = Path.cwd()
live = False
decades = range(1970, 2020, 10)
rating_min = 7.0
votes_min = 1000
base = folder / "01_raw/tmdb_candidates"
latest = base / "latest_run.txt"
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") if live else latest.read_text().strip()
run = base / run_id
run.mkdir(parents=True, exist_ok=True)
headers = {"User-Agent": "ADS206-film-study/0.1 (educational)", "Accept": "application/json"}
proxy_url = None
if live:
    settings = {}
    for line in (folder / ".env").read_text(encoding="utf-8-sig").splitlines():
        if "=" in line and not line.lstrip().startswith("#"):
            key, value = line.split("=", 1)
            settings[key.strip()] = value.strip().strip("\"'")
    proxy_url = settings.get("HTTPS_PROXY")
    token = settings.get("TMDB_READ_TOKEN", "")
    assert token and not any(char.isspace() for char in token), "Check the local token field."
    headers["Authorization"] = "Bearer " + token
print("Mode:", "live requests" if live else "saved responses")

Mode: saved responses


## 1. Request and save

`requests.get` sends the web request. Keep each response and its log. Pause between requests; retry temporary failures up to three times. Headers and the token are never logged.

If your network needs a proxy, the optional `HTTPS_PROXY` line in `.env` supplies it for these requests. No system setting is changed.

In [3]:
def fetch(params, name, url="https://api.themoviedb.org/3/discover/movie"):
    if not live:
        return json.loads((run / name).read_bytes())
    for attempt in range(3):
        reply = None
        clock = time.perf_counter()
        log = {"time_utc": datetime.now(timezone.utc).isoformat(),
               "url": url, "params": params,
               "retry_count": attempt, "collector": "AI-assisted candidate check"}
        try:
            reply = requests.get(log["url"], params=params, headers=headers, timeout=(10, 30),
                                 proxies={"https": proxy_url} if proxy_url else None)
            assert token.encode() not in reply.content, "Unexpected credential in response; not saved."
            tag = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
            file = f"{Path(name).stem}_{tag}.json"
            (run / file).write_bytes(reply.content)
            log.update(url=reply.url, status=reply.status_code, file=file,
                       redirects=[r.status_code for r in reply.history],
                       content_type=reply.headers.get("Content-Type"),
                       retry_after=reply.headers.get("Retry-After"),
                       sha256=hashlib.sha256(reply.content).hexdigest())
        except requests.RequestException as error:
            log["error"] = type(error).__name__
        log["seconds"] = round(time.perf_counter() - clock, 3)
        with (run / "requests.jsonl").open("a", encoding="utf-8") as file:
            file.write(json.dumps(log) + "\n")
        time.sleep(2)
        if reply is not None and reply.ok:
            (run / name).write_bytes(reply.content)
            return reply.json()
        if reply is not None and reply.status_code != 429 and reply.status_code < 500:
            raise RuntimeError(f"HTTP {reply.status_code}. See the request log.")
        if attempt == 2:
            raise ConnectionError("Request failed. See requests.jsonl.")
        delay = 2 ** attempt
        if reply is not None and reply.headers.get("Retry-After"):
            wait = reply.headers["Retry-After"]
            if not wait.isdigit() or int(wait) > 60:
                raise RuntimeError("Stop and review Retry-After before more requests.")
            delay = max(delay, int(wait))
        time.sleep(delay)

## 2. Read one page per decade

Use primary release dates, rating >= 7.0, vote count >= 1,000 and `vote_count.desc`. Read page 1 for each decade. No country or genre cutoff is applied. English labels do not restrict production country.

Only returned records count as collected data. The API-reported totals are saved but not treated as verified coverage estimates. This vote-ranked list is a purposive candidate frame, not a random sample.

In [4]:
records, checks = [], []
for decade in decades:
    params = {"primary_release_date.gte": f"{decade}-01-01",
              "primary_release_date.lte": f"{decade + 9}-12-31",
              "sort_by": "vote_count.desc", "page": 1, "language": "en-US",
              "include_adult": "false", "include_video": "false",
              "vote_average.gte": rating_min, "vote_count.gte": votes_min}
    data = fetch(params, f"discover_{decade}.json")
    assert data["page"] == 1 and isinstance(data["results"], list)
    rows = data["results"]
    assert len({row["id"] for row in rows}) == len(rows), "Repeated ID on this page."
    assert all(decade <= int(row["release_date"][:4]) < decade + 10 for row in rows), "Check dates."
    assert all(isinstance(row["vote_count"], int) and isinstance(row["vote_average"], (int, float))
               for row in rows), "Check rating fields."
    assert all(row["vote_average"] >= rating_min and row["vote_count"] >= votes_min for row in rows)
    assert all(a["vote_count"] >= b["vote_count"] for a, b in zip(rows, rows[1:])), "Check sorting."
    checks.append({"decade": f"{decade}s", "page_read": 1, "returned": len(rows),
                   "api_reported_total": data["total_results"], "api_reported_pages": data["total_pages"],
                   "blank_overviews": sum(not row.get("overview", "").strip() for row in rows)})
    for rank, row in enumerate(rows, 1):
        records.append({"tmdb_id": row["id"], "title": row["title"],
            "original_title": row["original_title"], "release_date": row["release_date"],
            "decade": f"{decade}s", "page": 1, "rank_on_page": rank,
            "rating": row["vote_average"], "vote_count": row["vote_count"],
            "original_language": row["original_language"], "genre_ids": row["genre_ids"],
            "overview": row.get("overview", ""), "selected_for_main_analysis": False,
            "tmdb_url": f"https://www.themoviedb.org/movie/{row['id']}"})
    print(f"{decade}s: read {len(rows)} candidates")
assert len({row["tmdb_id"] for row in records}) == len(records), "Repeated ID across decades."
view = pd.DataFrame(records)
pd.DataFrame(checks)[["decade", "page_read", "returned", "blank_overviews"]]

1970s: read 20 candidates
1980s: read 20 candidates
1990s: read 20 candidates
2000s: read 20 candidates
2010s: read 20 candidates


,decade,page_read,returned,blank_overviews
0,1970s,1,20,0
1,1980s,1,20,0
2,1990s,1,20,0
3,2000s,1,20,0
4,2010s,1,20,0


## 3. Read film details

Use each TMDB ID to request IMDb ID, runtime, all production countries and genre names. Keep the full response. These fields support later identity and full-film checks; they do not prove that a Cinemetrics measurement covers the whole film.

The credits endpoint supplies crew entries. Keep Director names for later title/year/director checks.

In [5]:
for number, row in enumerate(records, 1):
    uid = row["tmdb_id"]
    item = fetch({"language": "en-US"}, f"movie_{uid}.json",
                 f"https://api.themoviedb.org/3/movie/{uid}")
    assert item["id"] == uid
    row.update(imdb_id=item.get("imdb_id"), runtime_minutes=item.get("runtime"),
               countries=[country["name"] for country in item.get("production_countries", [])],
               country_codes=[country["iso_3166_1"] for country in item.get("production_countries", [])],
               genres=[genre["name"] for genre in item.get("genres", [])],
               title_details=item.get("title"), release_date_details=item.get("release_date"),
               rating_details=item.get("vote_average"), vote_count_details=item.get("vote_count"),
               overview_details=item.get("overview", ""),
               cinemetrics_check="pending identity and full-film review")
    credits = fetch({"language": "en-US"}, f"credits_{uid}.json",
                    f"https://api.themoviedb.org/3/movie/{uid}/credits")
    assert credits["id"] == uid
    row["directors"] = [person["name"] for person in credits.get("crew", []) if person.get("job") == "Director"]
    row["director_ids"] = [person["id"] for person in credits.get("crew", []) if person.get("job") == "Director"]
    if number % 10 == 0:
        print(f"Film details read: {number}/{len(records)}", flush=True)

Film details read: 10/100


Film details read: 20/100


Film details read: 30/100


Film details read: 40/100


Film details read: 50/100


Film details read: 60/100


Film details read: 70/100


Film details read: 80/100


Film details read: 90/100


Film details read: 100/100


## 4. Apply the collection standard

| Check | Rule |
| --- | --- |
| Dates | Primary release date from 1970-01-01 to 2019-12-31 |
| Candidate source | TMDB; the same query and ordering for every decade |
| Rating and votes | TMDB rating >= 7.0 and vote count >= 1,000; confirmed by the student |
| Page budget | One page per decade for this check; not a final sample quota |
| Geography | No country restriction; later record all production countries |
| Film scope | Feature films; exclude TV episodes, shorts, clips and excerpts |
| Identity | Check shared IMDb ID and year; otherwise review title, year and director |
| Measurements | A full-film Cinemetrics measurement is needed for shot-length analysis |
| Duplicates | One film in the final film table; keep repeat measurements for review |
| Missing evidence | Keep the candidate pending; do not fill missing shot data with guesses |

Ratings are TMDB audience scores. Vote counts count ratings, not viewers. English labels and original language do not establish production country. A first-page list overrepresents heavily rated films.

Apply the same rules to every decade. Do not choose or change films because of their shot-length results. Numerical cutoffs were confirmed by the student. Final sample size is not yet fixed.

In [6]:
for row in records:
    row["rating_and_votes_check"] = "pending cutoffs" if rating_min is None or votes_min is None else (
        "pass" if row["rating"] >= rating_min and row["vote_count"] >= votes_min else "below cutoff")
result = {"stage": "candidate collection; identity and full-film checks pending", "run_id": run_id,
          "source": "TMDB discover/movie", "sort_by": "vote_count.desc",
          "date_start": "1970-01-01", "date_end": "2019-12-31",
          "rating_cutoff": rating_min, "vote_count_cutoff": votes_min,
          "rules_status": "pending numerical cutoffs" if rating_min is None or votes_min is None else "cutoffs confirmed by student",
          "pages_per_decade": 1, "country_restriction": None,
          "required_scope": "feature film and full-film Cinemetrics measurement",
          "collector": "AI-assisted acquisition; numerical cutoffs confirmed by student",
          "final_sample_size": None, "checks": checks, "records": records}
if live:
    body = json.dumps(result, ensure_ascii=False, indent=2)
    (run / "candidates.json").write_text(body, encoding="utf-8")
    (folder / "02_audit/tmdb_candidates.json").write_text(body, encoding="utf-8")
    latest.write_text(run_id, encoding="utf-8")
view = pd.DataFrame(records)
view[["decade", "title", "rating", "vote_count", "runtime_minutes", "countries", "rating_and_votes_check"]]

,decade,title,rating,vote_count,runtime_minutes,countries,rating_and_votes_check
0,1970s,The Godfather,8.689,23952,175,[United States of America],pass
1,1970s,Star Wars,8.200,23146,121,[United States of America],pass
2,1970s,Alien,8.176,17101,117,"[United States of America, United Kingdom]",pass
3,1970s,The Godfather Part II,8.572,14594,202,[United States of America],pass
4,1970s,A Clockwork Orange,8.167,14299,137,"[United Kingdom, United States of America]",pass
...,...,...,...,...,...,...,...
95,2010s,Spider-Man: Homecoming,7.300,24221,133,[United States of America],pass
96,2010s,Black Panther,7.359,24220,135,[United States of America],pass
97,2010s,Doctor Strange,7.425,24188,115,[United States of America],pass
98,2010s,Inside Out,7.906,24146,95,[United States of America],pass


## Credits

<a href="https://www.themoviedb.org"><img src="https://www.themoviedb.org/assets/2/v4/logos/v2/blue_short-8e7b30f73a4020692ccca9c88bafe5dcb6f8a62a4c6bc55cd9ba82bb2cd95f6c.svg" width="120" alt="TMDB"></a>

This product uses the TMDB API but is not endorsed or certified by TMDB.

[TMDB API documentation](https://developer.themoviedb.org/reference/discover-movie) | [TMDB attribution](https://www.themoviedb.org/about/logos-attribution)

Codex assisted with acquisition code and replay checks. Final selection rules and interpretation need student review.